# DATA EXTRACTION FROM MIMIC-III FOR MetaLearning4EHRs

This file is for extracting heart-related diseases informations(ICD9 Code start with 42)

In [ ]:
# Import libraries
import pandas as pd
import psycopg2
import os
from pathlib import Path

In [ ]:
# Update connection details to eICU
conn = psycopg2.connect("dbname=eicu user=postgres password=postgres host=localhost port=5432")

# Update the path for data extraction here
export_dir=r"data\eICU_first24h\all"
Path(export_dir).mkdir(exist_ok=True, parents=True)

## Extract considered icd9 code

icd9_code start with V and E are not considered

In [ ]:
query = """
SELECT *
FROM (
	SELECT 
		SPLIT_PART(icd9code, ',', 1) AS icd9_code,
        diagnosisstring,
	    COUNT(DISTINCT patientunitstayid) AS count
	FROM diagnosis
	WHERE diagnosispriority = 'Primary'
		AND icd9code IS NOT NULL
	    AND icd9code <> ''
		AND diagnosisoffset < 24*60 
		GROUP BY icd9code, diagnosisstring
	) d
WHERE d.icd9_code NOT LIKE 'V%' 
	AND d.icd9_code NOT LIKE 'E%'
ORDER BY count DESC;
"""
df = pd.read_sql_query(query, conn, dtype={'icd9_code': str})
df.info()

## Demography

Age, Gender, ethnicity<br>
16 < Age < 90, in_hospital_expire_flag = 0

In [ ]:
query = f"""
WITH ranked_rows AS (
    SELECT 
        p.uniquePid AS subject_id, 
        p.patientUnitStayID AS stay_id, 
        p.hospitalID, 
        p.ethnicity, 
        p.gender, 
        p.age, 
        p.unitDischargeOffset, 
        p.unitDischargeOffset AS icustaytime, -- Stay time  
        p.hospitaldischargeoffset AS hospdischtime, 
        d.icd9code,
        ROW_NUMBER() OVER (PARTITION BY p.patientUnitStayID, d.icd9code ORDER BY p.unitDischargeOffset DESC) AS row_num
    FROM patient p 
    LEFT JOIN diagnosis d ON d.patientUnitStayID = p.patientUnitStayID
    WHERE NOT p.age = '> 89'
        AND NOT p.age = ''
        AND p.unitDischargeOffset >= 24*60 -- ICU stay time >= 24h
        AND CAST(p.age AS INT) > 16 AND CAST(p.age AS INT) < 90
        AND d.diagnosisoffset <= 24*60
        AND d.diagnosispriority = 'Primary'
        AND d.icd9code IS NOT NULL
        AND d.icd9code <> ''
)
SELECT 
    subject_id, 
    stay_id, 
    hospitalID, 
    ethnicity, 
    gender, 
    age, 
    unitDischargeOffset, 
    icustaytime, 
    hospdischtime, 
    icd9code
FROM ranked_rows
WHERE row_num = 1;
"""
df = pd.read_sql_query(query, conn)
df.info()

considered_stay_id = df['stay_id'].tolist()
considered_stay_id_str = ', '.join(map(str, considered_stay_id))

### APACHE IV and IVa 
IV and IVa have same APACHE score but different parameters to calculate the mortality probability

In [ ]:
query = f"""
SELECT apachepatientresultsid, patientunitstayid, apachescore, apacheversion, predictedicumortality
FROM apachePatientResult
WHERE patientunitstayid in ({considered_stay_id_str});
"""
df = pd.read_sql_query(query,conn)

# Pivot so that each apacheversion becomes a column
df_pivot = df.pivot(index=('patientunitstayid', 'apachescore'), 
                    columns='apacheversion', 
                    values='predictedicumortality')

# Rename the columns
df_pivot = df_pivot.rename(columns={
    'IV': 'icumortality_apache_iv',
    'IVa': 'icumortality_apache_iva'
}).reset_index()

df_pivot = df_pivot.rename(columns={
    'patientunitstayid': "stay_id"
})
# Save to CSV
df_pivot.to_csv(os.path.join(export_dir, "apache_score.csv"), index=False, sep=',')

# Check structure
df_pivot.info()